# Build training_pairs.pkl (Laptop, RTX 4050)

For every training row:
- Retrieve top-K candidate **answers** via BGE-M3 (same-subset, self-excluded).
- Score each candidate against the row's reference with ROUGE-1.
- Save (query, candidate, rouge_target) triples to `training_pairs.pkl`.

Output feeds directly into `train_reranker_laptop.ipynb`.

Expected wall-clock on RTX 4050: ~15-25 minutes total.

## 1 — Environment setup (single GPU + expandable allocator)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, pickle, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.neighbors import NearestNeighbors
from rouge_score import rouge_scorer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print(f"  {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("  CUDA not available — encoding will run on CPU and be very slow.")
    print("  Check your torch install with: python -c \"import torch; print(torch.cuda.is_available())\"")


ModuleNotFoundError: No module named 'pandas'

## 2 — Config

In [ ]:
HERE       = Path(".").resolve()
TRAIN_CSV  = HERE / "Train.csv"
OUT_PATH   = HERE / "training_pairs.pkl"

QCOL, ACOL, GCOL = "input", "output", "subset"
BI_MODEL_NAME = "BAAI/bge-m3"
K             = 10            # candidates per training row
ENCODE_BATCH  = 32            # bi-encoder batch — fits 6 GB
SEED          = 42

## 3 — Load train and inspect

In [ ]:
def load_csv(path):
    df = pd.read_csv(path).dropna(subset=["input","output","subset"])
    for c in [QCOL, GCOL]:
        df[c] = df[c].fillna("").astype(str).str.strip()
    df[ACOL] = df[ACOL].fillna("").astype(str).str.strip()
    df = df[(df[QCOL]!="") & (df[ACOL]!="")]
    return df.reset_index(drop=True)

train_df = load_csv(TRAIN_CSV)
print(f"Loaded {len(train_df)} train rows")
print(train_df[GCOL].value_counts().to_string())

## 4 — Encode train questions with BGE-M3

Loads BGE-M3 (~2.2 GB on GPU), encodes per-subset, builds NN indices with K+1
neighbors so we can drop self-matches and still keep K candidates.

In [ ]:
from sentence_transformers import SentenceTransformer

def gpu_status(label=""):
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"  [GPU {label}] {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")

gpu_status("before BGE-M3")
print(f"\nLoading {BI_MODEL_NAME} ...")
bi = SentenceTransformer(BI_MODEL_NAME, device=DEVICE)
gpu_status("after BGE-M3 load")

def encode_subset_indices(df, encoder, k):
    indices = {}
    for g, grp in df.groupby(GCOL):
        embs = encoder.encode(grp[QCOL].tolist(),
                              normalize_embeddings=True,
                              show_progress_bar=False,
                              batch_size=ENCODE_BATCH,
                              convert_to_numpy=True)
        n_fit = min(k + 1, len(grp))
        nn_ = NearestNeighbors(n_neighbors=n_fit, metric="cosine").fit(embs)
        indices[g] = {
            "nn":  nn_,
            "embs": embs,
            "ans": np.array(grp[ACOL].astype(str).tolist(), dtype=object),
            "orig_idx": np.array(grp.index.tolist()),
        }
        print(f"  {g}: {len(grp)} rows indexed")
    return indices

print(f"\nEncoding train questions (K={K}) ...")
t0 = time.time()
indices = encode_subset_indices(train_df, bi, K)
print(f"  done in {(time.time()-t0)/60:.1f} min")

## 5 — Release BGE-M3 from GPU before the CPU ROUGE pass

Frees ~2 GB so the next stage doesn't fight for memory.

In [ ]:
bi.to("cpu")
del bi
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
gpu_status("after BGE-M3 release")

## 6 — Build pairs with ROUGE-1 labels

For every row: top-K candidates (self excluded by original index), each scored
against the row's reference answer with the competition's exact whitespace ROUGE.

In [ ]:
class WhitespaceTokenizer:
    def tokenize(self, t): return [] if t is None else str(t).strip().split()
_SCORER = rouge_scorer.RougeScorer(["rouge1"], tokenizer=WhitespaceTokenizer(), use_stemmer=False)
def rouge1(p, r): return _SCORER.score(str(r), str(p))["rouge1"].fmeasure

def build_pairs(train_df, indices, k):
    pairs_q, pairs_c, pairs_y = [], [], []
    total = len(train_df); seen = 0
    t0 = time.time(); last_print = t0
    for g, grp in train_df.groupby(GCOL):
        m = indices[g]
        _, idx = m["nn"].kneighbors(m["embs"], n_neighbors=min(k+1, len(m["ans"])))
        for local_i, (orig_idx, row) in enumerate(grp.iterrows()):
            ref = str(row[ACOL]); q = str(row[QCOL]); picked = 0
            for j in idx[local_i]:
                if m["orig_idx"][j] == orig_idx:    # self-exclusion
                    continue
                cand = str(m["ans"][j])
                pairs_q.append(q); pairs_c.append(cand)
                pairs_y.append(rouge1(cand, ref))
                picked += 1
                if picked == k: break
            seen += 1
            now = time.time()
            if now - last_print > 5:
                rate = seen / (now - t0)
                eta = (total - seen) / rate if rate > 0 else 0
                print(f"  {seen}/{total} rows ({rate:.1f}/s, ETA {eta/60:.1f} min)")
                last_print = now
    return pairs_q, pairs_c, np.array(pairs_y, dtype=np.float32)

tr_q, tr_c, tr_y = build_pairs(train_df, indices, K)
print(f"\nBuilt {len(tr_q)} pairs")

## 7 — Sanity checks

In [ ]:
print("Target ROUGE-1 distribution:")
print(pd.Series(tr_y).describe().round(4))
print(f"Targets >= 0.5: {int((tr_y>=0.5).sum())} ({(tr_y>=0.5).mean()*100:.1f}%)")
print(f"Targets >= 0.9: {int((tr_y>=0.9).sum())} ({(tr_y>=0.9).mean()*100:.1f}%)")

# Self-leakage check — must be 0
own_answer = train_df.set_index(QCOL)[ACOL].astype(str).to_dict()
leak = sum(1 for q, c in zip(tr_q, tr_c) if own_answer.get(q) == c)
print(f"\nSelf-leakage (cand == row's own answer): {leak}  (must be 0)")

## 8 — Save

In [ ]:
with open(OUT_PATH, "wb") as f:
    pickle.dump({"tr_q": tr_q, "tr_c": tr_c, "tr_y": tr_y}, f)
print(f"Saved {OUT_PATH.name} ({OUT_PATH.stat().st_size/1e6:.1f} MB)")
print("Open train_reranker_laptop.ipynb next.")